In [1]:
import os 

# set directory to parent directory
os.chdir("..")

# print current working directory
print("Current Working Directory:", os.getcwd())


Current Working Directory: /cluster/work/vogtlab/Group/jogoncalves/treevae


In [2]:

from FID.fid_score import calculate_fid, get_precomputed_fid_scores_path, save_fid_stats_as_dict
import gzip
import numpy as np
import os
from PIL import Image
import pickle
import torch
import yaml
import argparse
from pathlib import Path
from utils.data_utils import get_data, get_gen
from utils.utils import reset_random_seeds, prepare_config
from models.losses import loss_reconstruction_binary, loss_reconstruction_mse
from concurrent.futures import ThreadPoolExecutor, as_completed

In [3]:
def load_image(file_path):
    img = Image.open(file_path)
    return np.array(img)

def load_images_from_path(path):
    images = []
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(load_image, os.path.join(path, filename))
                   for filename in os.listdir(path) if filename.endswith(".png")]
        for future in as_completed(futures):
            images.append(future.result())
    return np.array(images)

In [4]:
path = '/Users/jorgegoncalves/Library/CloudStorage/OneDrive-Persönlich/Dokumente/Universität/Master/HS23/Master_Thesis/Code/treevae/models/experiments/'

dataset = 'mnist/'
ex_name = '20240221-150914_9e40a'

dataset = 'cifar10/'
ex_name = '20240303-144417_f2db4'

checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)
print(configs)

from utils.data_utils import get_data, get_gen

# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "mps"

FileNotFoundError: [Errno 2] No such file or directory: '/Users/jorgegoncalves/Library/CloudStorage/OneDrive-Persönlich/Dokumente/Universität/Master/HS23/Master_Thesis/Code/treevae/models/experiments/cifar10/20240303-144417_f2db4/config.yaml'

In [17]:
ddpm_samples_path = '../from_cluster/results/' + dataset + 'seed_1/ddpm/sample/'
ddpm_reconstructions_path = '../from_cluster/results/' + dataset + 'seed_1/ddpm/recons/'

vae_samples_path = '../from_cluster/results/' + dataset + 'seed_1/vae/sample/'
vae_reconstructions_path = '../from_cluster/results/' + dataset + 'seed_1/vae/recons/'

In [18]:
# load all images from the path and create a numpy array
ddpm_samples = []
for filename in os.listdir(ddpm_samples_path):
    if filename.endswith(".png"):
        img = Image.open(ddpm_samples_path + filename)
        img = np.array(img)
        ddpm_samples.append(img)
ddpm_samples = np.array(ddpm_samples)
print("Nb. of ddpm samples: ", len(ddpm_samples), "\n")

ddpm_reconstructions = []
for filename in os.listdir(ddpm_reconstructions_path):
    if filename.endswith(".png"):
        img = Image.open(ddpm_reconstructions_path + filename)
        img = np.array(img)
        ddpm_reconstructions.append(img)
ddpm_reconstructions = np.array(ddpm_reconstructions)
print("Nb. of ddpm reconstructions: ", len(ddpm_reconstructions), "\n")

vae_samples = []
for filename in os.listdir(vae_samples_path):
    if filename.endswith(".png"):
        img = Image.open(vae_samples_path + filename)
        img = np.array(img)
        vae_samples.append(img)
vae_samples = np.array(vae_samples)
print("Nb. of vae samples: ", len(vae_samples), "\n")

vae_reconstructions = []
for filename in os.listdir(vae_reconstructions_path):
    if filename.endswith(".png"):
        img = Image.open(vae_reconstructions_path + filename)
        img = np.array(img)
        vae_reconstructions.append(img)
vae_reconstructions = np.array(vae_reconstructions)
print("Nb. of vae reconstructions: ", len(vae_reconstructions), "\n")



Nb. of ddpm samples:  10000 

Nb. of ddpm reconstructions:  10000 

Nb. of vae samples:  10000 

Nb. of vae reconstructions:  10000 



In [19]:
# precompute or load fid scores for train and test
data_stats_train = get_precomputed_fid_scores_path(trainset.dataset.data, configs['data']['data_name'],
                                                subset="train", device=device)
data_stats_test = get_precomputed_fid_scores_path(testset.dataset.data, configs['data']['data_name'], subset="test",
                                                device=device)

---

VAE SAMPLES

In [20]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(vae_samples, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)

Saving FID statistics


100%|██████████| 40/40 [00:52<00:00,  1.30s/it]


FID score for generated images compared to train set: 207.80528852902063
FID score for generated images compared to test set: 208.84495887358773


DDPM SAMPLES

In [21]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(ddpm_samples, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)


Saving FID statistics


100%|██████████| 40/40 [00:52<00:00,  1.31s/it]


FID score for generated images compared to train set: 20.767625425869255
FID score for generated images compared to test set: 22.80768232044005


---

VAE RECONSTRUCTIONS

In [22]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(vae_reconstructions, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)

Saving FID statistics


100%|██████████| 40/40 [00:51<00:00,  1.30s/it]


FID score for generated images compared to train set: 193.19771977090804
FID score for generated images compared to test set: 194.26382854522302


DDPM RECONSTRUCTIONS

In [15]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(ddpm_reconstructions, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)

Saving FID statistics


100%|██████████| 40/40 [00:52<00:00,  1.31s/it]


FID score for generated images compared to train set: 1.0107779633030702
FID score for generated images compared to test set: 1.459633255183519


---
---

# MNIST

In [11]:

dataset = 'mnist/'

# use config of some trained mnist model to load data correctly
ex_name = '20240906-142239_b81c6'
path = '/cluster/work/vogtlab/Group/jogoncalves/treevae/models/experiments/'

checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)

# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

# setup device, mps is for M1 or M2 macs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "mps"

# precompute or load fid scores for train and test
data_stats_train = get_precomputed_fid_scores_path(trainset.dataset.data, configs['data']['data_name'],
                                                subset="train", device=device)
data_stats_test = get_precomputed_fid_scores_path(testset.dataset.data, configs['data']['data_name'], subset="test",
                                                device=device)

# paths to the generated images
base_path = '../results_ICLR/'
exp_path = 'cond_on_path'
seed_list = ["seed_1", "seed_2", "seed_3", "seed_4", "seed_5", "seed_6", "seed_7", "seed_8", "seed_9", "seed_10"]

# lists to store FID scores for each seed
mnist_train_FID_generations_ddpm = []
mnist_test_FID_generations_ddpm = []
mnist_train_FID_reconstructions_ddpm = []
mnist_test_FID_reconstructions_ddpm = []

for i, seed in enumerate(seed_list):
    ddpm_samples_path = base_path + dataset + exp_path + "/ddim/" + seed + '/ddpm/sample/'
    ddpm_reconstructions_path = base_path + dataset + exp_path + "/ddim/" + seed + '/ddpm/recons/'

    # load all images from the path and create a numpy array
    ddpm_samples = load_images_from_path(ddpm_samples_path)
    ddpm_reconstructions = load_images_from_path(ddpm_reconstructions_path)

    # precompute FID scores for generated and reconstructed images of DDPM

    # DDPM samples
    ddpm_stats_generations = save_fid_stats_as_dict(ddpm_samples, batch_size=256, device=device, dims=2048)
    train_FID_gen_ddpm = calculate_fid([data_stats_train, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    test_FID_gen_ddpm = calculate_fid([data_stats_test, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    mnist_train_FID_generations_ddpm.append(train_FID_gen_ddpm)
    mnist_test_FID_generations_ddpm.append(test_FID_gen_ddpm)

    # DDPM reconstructions
    ddpm_stats_reconstructions = save_fid_stats_as_dict(ddpm_reconstructions, batch_size=256, device=device, dims=2048)
    train_FID_recon_ddpm = calculate_fid([data_stats_train, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    test_FID_recon_ddpm = calculate_fid([data_stats_test, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    mnist_train_FID_reconstructions_ddpm.append(train_FID_recon_ddpm)
    mnist_test_FID_reconstructions_ddpm.append(test_FID_recon_ddpm)



Saving FID statistics


100%|██████████| 40/40 [00:13<00:00,  3.02it/s]


Saving FID statistics


100%|██████████| 40/40 [00:12<00:00,  3.14it/s]


Saving FID statistics


...

100%|██████████| 40/40 [00:12<00:00,  3.14it/s]


Saving FID statistics


100%|██████████| 40/40 [00:12<00:00,  3.12it/s]


Saving FID statistics


100%|██████████| 40/40 [00:12<00:00,  3.14it/s]


In [12]:
print("\nFID scores for DDPM samples compared to train set:\n", mnist_train_FID_generations_ddpm)
print("\nFID scores for DDPM samples compared to test set:\n", mnist_test_FID_generations_ddpm)
print("\nFID scores for DDPM reconstructions compared to train set:\n", mnist_train_FID_reconstructions_ddpm)
print("\nFID scores for DDPM reconstructions compared to test set:\n", mnist_test_FID_reconstructions_ddpm)


FID scores for DDPM samples compared to train set:
 [1.0144767302461446, 1.1526323538118675, 1.0592236816090121, 1.033344724954361, 1.0543989557779696, 1.0937484486744324, 1.0467303466982116, 1.0617024238485158, 1.0123680750395465, 1.0973232213307824]

FID scores for DDPM samples compared to test set:
 [1.7218031715424331, 1.91921332694821, 1.8136703445894113, 1.6689971091697657, 1.7839037918941472, 1.7466216705165607, 1.7841780721453233, 1.7221718136240725, 1.6946479838632058, 1.8821913138930881]

FID scores for DDPM reconstructions compared to train set:
 [1.2955353299931858, 1.2213884358837674, 1.2770245961161208, 1.2487025033992438, 1.2124604336690936, 1.3061391013405057, 1.182246762538142, 1.2620607912588184, 1.2277112273789612, 1.1871781431350428]

FID scores for DDPM reconstructions compared to test set:
 [1.4698345436933096, 1.4356182643412865, 1.4800373680263021, 1.4553337492199319, 1.4401319498838347, 1.4931309754894357, 1.430637012695371, 1.446069930518604, 1.43686455355651

In [13]:
# compute average and standard deviation of FID scores for DDPM 
print("MNIST")
print("-----------------------------------")
print("Reconstructions")
print("\nAverage FID score for DDPM reconstructions compared to test set:", np.mean(mnist_test_FID_reconstructions_ddpm))
print("\nStandard deviation FID score for DDPM reconstructions compared to test set:", np.std(mnist_test_FID_reconstructions_ddpm))
print("-----------------------------------")
print("Generations")
print("\nAverage FID score for DDPM samples compared to test set:", np.mean(mnist_test_FID_generations_ddpm))
print("\nStandard deviation FID score for DDPM samples compared to test set:", np.std(mnist_test_FID_generations_ddpm))

MNIST
-----------------------------------
Reconstructions

Average FID score for DDPM reconstructions compared to test set: 1.4506697481595296

Standard deviation FID score for DDPM reconstructions compared to test set: 0.022373188253394388
-----------------------------------
Generations

Average FID score for DDPM samples compared to test set: 1.7737398598186218

Standard deviation FID score for DDPM samples compared to test set: 0.07619028859342365


# FashionMNIST

In [14]:

dataset = 'fmnist/'

# use config of some trained fmnist model to load data correctly
ex_name = '20240906-144022_3c565'
path = '/cluster/work/vogtlab/Group/jogoncalves/treevae/models/experiments/'

checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)

# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

# setup device, mps is for M1 or M2 macs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "mps"

# precompute or load fid scores for train and test
data_stats_train = get_precomputed_fid_scores_path(trainset.dataset.data, configs['data']['data_name'],
                                                subset="train", device=device)
data_stats_test = get_precomputed_fid_scores_path(testset.dataset.data, configs['data']['data_name'], subset="test",
                                                device=device)

# paths to the generated images
base_path = '../results_ICLR/'
exp_path = 'cond_on_path'
seed_list = ["seed_1", "seed_2", "seed_3", "seed_4", "seed_5", "seed_6", "seed_7", "seed_8", "seed_9", "seed_10"]

# lists to store FID scores for each seed
fmnist_train_FID_generations_ddpm = []
fmnist_test_FID_generations_ddpm = []
fmnist_train_FID_reconstructions_ddpm = []
fmnist_test_FID_reconstructions_ddpm = []

for i, seed in enumerate(seed_list):
    ddpm_samples_path = base_path + dataset + exp_path + "/ddim/" + seed + '/ddpm/sample/'
    ddpm_reconstructions_path = base_path + dataset + exp_path + "/ddim/" + seed + '/ddpm/recons/'

    # load all images from the path and create a numpy array
    ddpm_samples = load_images_from_path(ddpm_samples_path)
    ddpm_reconstructions = load_images_from_path(ddpm_reconstructions_path)

    # precompute FID scores for generated and reconstructed images of VAE and DDPM

    # DDPM samples
    ddpm_stats_generations = save_fid_stats_as_dict(ddpm_samples, batch_size=256, device=device, dims=2048)
    train_FID_gen_ddpm = calculate_fid([data_stats_train, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    test_FID_gen_ddpm = calculate_fid([data_stats_test, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    fmnist_train_FID_generations_ddpm.append(train_FID_gen_ddpm)
    fmnist_test_FID_generations_ddpm.append(test_FID_gen_ddpm)

    # DDPM reconstructions
    ddpm_stats_reconstructions = save_fid_stats_as_dict(ddpm_reconstructions, batch_size=256, device=device, dims=2048)
    train_FID_recon_ddpm = calculate_fid([data_stats_train, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    test_FID_recon_ddpm = calculate_fid([data_stats_test, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    fmnist_train_FID_reconstructions_ddpm.append(train_FID_recon_ddpm)
    fmnist_test_FID_reconstructions_ddpm.append(test_FID_recon_ddpm)

Saving FID statistics


100%|██████████| 40/40 [00:12<00:00,  3.15it/s]


Saving FID statistics


100%|██████████| 40/40 [00:12<00:00,  3.14it/s]


Saving FID statistics


...

100%|██████████| 40/40 [00:12<00:00,  3.12it/s]


Saving FID statistics


100%|██████████| 40/40 [00:12<00:00,  3.13it/s]


Saving FID statistics


100%|██████████| 40/40 [00:12<00:00,  3.14it/s]


In [15]:
print("\nFID scores for DDPM samples compared to train set:\n", fmnist_train_FID_generations_ddpm)
print("\nFID scores for DDPM samples compared to test set:\n", fmnist_test_FID_generations_ddpm)
print("\nFID scores for DDPM reconstructions compared to train set:\n", fmnist_train_FID_reconstructions_ddpm)
print("\nFID scores for DDPM reconstructions compared to test set:\n", fmnist_test_FID_reconstructions_ddpm)


FID scores for DDPM samples compared to train set:
 [4.210457084134305, 4.524748159982437, 4.50707660796968, 3.9934355679473015, 3.6441726017349083, 4.361265233720246, 4.047928469418139, 4.050324395568225, 5.136324750428116, 4.069361800338811]

FID scores for DDPM samples compared to test set:
 [5.345420097886631, 5.640318081244686, 5.683912421382729, 5.112825026460257, 4.7857510349612085, 5.500723328246636, 5.2237418397353395, 5.164570186687911, 6.332425347443689, 5.192675752085336]

FID scores for DDPM reconstructions compared to train set:
 [4.526800099538548, 4.6710858047538295, 4.5270373050563535, 4.360705066755429, 3.8436768641153094, 4.264030218649907, 4.575280393428898, 4.079523606619432, 5.950296906082485, 4.136346847070968]

FID scores for DDPM reconstructions compared to test set:
 [5.574419324543271, 5.75619988957277, 5.584982010120143, 5.400013519085178, 4.8654475972018645, 5.277484736017357, 5.640776718134191, 5.1224435580001, 7.060862814781103, 5.163635591588786]


In [16]:
# compute average and standard deviation of FID scores for DDPM 
print("FashionMNIST")
print("-----------------------------------")
print("Reconstructions")
print("\nAverage FID score for DDPM reconstructions compared to test set:", np.mean(fmnist_test_FID_reconstructions_ddpm))
print("\nStandard deviation FID score for DDPM reconstructions compared to test set:", np.std(fmnist_test_FID_reconstructions_ddpm))
print("-----------------------------------")
print("Generations")
print("\nAverage FID score for DDPM samples compared to test set:", np.mean(fmnist_test_FID_generations_ddpm))
print("\nStandard deviation FID score for DDPM samples compared to test set:", np.std(fmnist_test_FID_generations_ddpm))

FashionMNIST
-----------------------------------
Reconstructions

Average FID score for DDPM reconstructions compared to test set: 5.544626575904476

Standard deviation FID score for DDPM reconstructions compared to test set: 0.5685464829370955
-----------------------------------
Generations

Average FID score for DDPM samples compared to test set: 5.398236311613442

Standard deviation FID score for DDPM samples compared to test set: 0.4012627389445423


# CIFAR10

In [11]:
dataset = 'cifar10/'

# use config of some trained cifar10 model to load data correctly
ex_name = '20240613-111829_7bc09'
path = '/cluster/work/vogtlab/Group/jogoncalves/treevae/models/experiments/'


checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)

# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

# setup device, mps is for M1 or M2 macs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "mps"

# precompute or load fid scores for train and test
data_stats_train = get_precomputed_fid_scores_path(trainset.dataset.data, configs['data']['data_name'],
                                                subset="train", device=device)
data_stats_test = get_precomputed_fid_scores_path(testset.dataset.data, configs['data']['data_name'], subset="test",
                                                device=device)

# paths to the generated images
base_path = '../results_ICLR/'
exp_path = 'cond_on_path'
seed_list = ["seed_1", "seed_2", "seed_3", "seed_4", "seed_5", "seed_6", "seed_7", "seed_8", "seed_9", "seed_10"]

# lists to store FID scores for each seed
cifar10_train_FID_generations_ddpm = []
cifar10_test_FID_generations_ddpm = []
cifar10_train_FID_reconstructions_ddpm = []
cifar10_test_FID_reconstructions_ddpm = []

for i, seed in enumerate(seed_list):
    ddpm_samples_path = base_path + dataset + exp_path + "/ddim/" + seed + '/ddpm/sample/'
    ddpm_reconstructions_path = base_path + dataset + exp_path + "/ddim/" + seed + '/ddpm/recons/'

    # load all images from the path and create a numpy array
    ddpm_samples = load_images_from_path(ddpm_samples_path)
    ddpm_reconstructions = load_images_from_path(ddpm_reconstructions_path)

    # precompute FID scores for generated and reconstructed images of DDPM

    # DDPM samples
    ddpm_stats_generations = save_fid_stats_as_dict(ddpm_samples, batch_size=256, device=device, dims=2048)
    train_FID_gen_ddpm = calculate_fid([data_stats_train, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    test_FID_gen_ddpm = calculate_fid([data_stats_test, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    cifar10_train_FID_generations_ddpm.append(train_FID_gen_ddpm)
    cifar10_test_FID_generations_ddpm.append(test_FID_gen_ddpm)

    # DDPM reconstructions
    ddpm_stats_reconstructions = save_fid_stats_as_dict(ddpm_reconstructions, batch_size=256, device=device, dims=2048)
    train_FID_recon_ddpm = calculate_fid([data_stats_train, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    test_FID_recon_ddpm = calculate_fid([data_stats_test, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    cifar10_train_FID_reconstructions_ddpm.append(train_FID_recon_ddpm)
    cifar10_test_FID_reconstructions_ddpm.append(test_FID_recon_ddpm)

Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified
Saving FID statistics


100%|██████████| 40/40 [00:13<00:00,  2.98it/s]


Saving FID statistics


100%|██████████| 40/40 [00:12<00:00,  3.13it/s]


Saving FID statistics


...

100%|██████████| 40/40 [00:12<00:00,  3.12it/s]


Saving FID statistics


100%|██████████| 40/40 [00:12<00:00,  3.13it/s]


Saving FID statistics


100%|██████████| 40/40 [00:12<00:00,  3.11it/s]


In [12]:
print("\nFID scores for DDPM samples compared to train set:\n", cifar10_train_FID_generations_ddpm)
print("\nFID scores for DDPM samples compared to test set:\n", cifar10_test_FID_generations_ddpm)
print("\nFID scores for DDPM reconstructions compared to train set:\n", cifar10_train_FID_reconstructions_ddpm)
print("\nFID scores for DDPM reconstructions compared to test set:\n", cifar10_test_FID_reconstructions_ddpm)


FID scores for DDPM samples compared to train set:
 [15.82907076256788, 16.145766139376974, 15.472436228580875, 15.97262838374894, 16.20801639100023, 16.43749416674808, 15.218083283731062, 15.73118537314781, 15.781212560662084, 15.003546167779234]

FID scores for DDPM samples compared to test set:
 [17.817548731891577, 18.18128913764025, 17.521947786705027, 18.018582670149442, 18.192306662012697, 18.471250010279277, 17.251809531402614, 17.754341499421685, 17.838664524434762, 17.0507517144992]

FID scores for DDPM reconstructions compared to train set:
 [10.341973335898729, 11.120580482441653, 10.610461758283236, 11.145119912113614, 10.209215757896288, 11.009723481420622, 10.413448688497681, 11.018246544747967, 10.8540143755896, 10.236969737936874]

FID scores for DDPM reconstructions compared to test set:
 [12.185462697889648, 12.945643886966536, 12.470263876054446, 12.99954365112535, 12.025675189640197, 12.832820681754015, 12.257871584642373, 12.863649560924102, 12.681056635046218, 1

In [13]:
# compute average and standard deviation of FID scores for DDPM 
print("CIFAR-10")
print("-----------------------------------")
print("Reconstructions")
print("\nAverage FID score for DDPM reconstructions compared to test set:", np.mean(cifar10_test_FID_reconstructions_ddpm))
print("\nStandard deviation FID score for DDPM reconstructions compared to test set:", np.std(cifar10_test_FID_reconstructions_ddpm))
print("-----------------------------------")
print("Generations")
print("\nAverage FID score for DDPM samples compared to test set:", np.mean(cifar10_test_FID_generations_ddpm))
print("\nStandard deviation FID score for DDPM samples compared to test set:", np.std(cifar10_test_FID_generations_ddpm))

CIFAR-10
-----------------------------------
Reconstructions

Average FID score for DDPM reconstructions compared to test set: 12.535793729782034

Standard deviation FID score for DDPM reconstructions compared to test set: 0.3546558138390279
-----------------------------------
Generations

Average FID score for DDPM samples compared to test set: 17.809849226843653

Standard deviation FID score for DDPM samples compared to test set: 0.4171039713858374


# CUBICC

In [5]:
dataset = 'cubicc/'

# use config of some trained cubicc model to load data correctly
ex_name = '20240923-013355_8ddcc'
path = '/cluster/work/vogtlab/Group/jogoncalves/treevae/models/experiments/'

checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)

In [6]:
configs["training"]

{'act_function': 'swish',
 'activation': 'mse',
 'aug_decisions_weight': 100,
 'augment': True,
 'augmentation_method': ['InfoNCE', 'instancewise_full'],
 'batch_size': 256,
 'bottom_up_channels': [128, 128, 128, 128, 128, 128, 128],
 'compute_fid': True,
 'compute_ll': False,
 'decay_kl': 0.01,
 'decay_lr': 0.1,
 'decay_stepsize': 100,
 'dim_mod_conv': True,
 'dropout_router': 0.0,
 'encoder': 'cnn2',
 'grow': True,
 'initial_depth': 1,
 'inp_channels': 3,
 'inp_shape': 64,
 'intermediate_fulltrain': True,
 'kl_start': 0.01,
 'latent_channels': [64, 64, 64, 64, 64, 64, 64],
 'lr': 0.001,
 'num_clusters_tree': 10,
 'num_epochs': 100,
 'num_epochs_finetuning': 0,
 'num_epochs_intermediate_fulltrain': 0,
 'num_epochs_smalltree': 100,
 'prune': True,
 'representation_dim': 4,
 'res_connections': True,
 'save_images': True,
 'spectral_norm': False,
 'weight_decay': 1e-05}

In [1]:
config_path = '/cluster/work/vogtlab/Group/jogoncalves/treevae/configs/cubicc.yml'

# Open the file using the built-in open function
with open(config_path, mode='r') as yamlfile:
    configs = yaml.safe_load(yamlfile)

NameError: name 'yaml' is not defined

In [ ]:
configs["ddpm"]["batch_size"] = 4
# load data
trainset, trainset_eval, testset = get_data(configs["ddpm"])
gen_train = get_gen(trainset, configs["ddpm"], validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs["ddpm"], validation=True, shuffle=False)
gen_test = get_gen(testset, configs["ddpm"], validation=True, shuffle=False)
#y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
#y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

# setup device, mps is for M1 or M2 macs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "mps"

In [ ]:
# precompute or load fid scores for train and test
data_stats_train = get_precomputed_fid_scores_path(trainset.dataset.data, configs['data']['data_name'],
                                                subset="train", device=device)
data_stats_test = get_precomputed_fid_scores_path(testset.dataset.data, configs['data']['data_name'], subset="test",
                                                device=device)

# paths to the generated images
base_path = '../results_ICLR/'
exp_path = 'cond_on_path'
seed_list = ["seed_1", "seed_2", "seed_3", "seed_4", "seed_5", "seed_6", "seed_7", "seed_8", "seed_9", "seed_10"]

# lists to store FID scores for each seed
cubicc_train_FID_generations_ddpm = []
cubicc_test_FID_generations_ddpm = []
cubicc_train_FID_reconstructions_ddpm = []
cubicc_test_FID_reconstructions_ddpm = []

for i, seed in enumerate(seed_list):
    ddpm_samples_path = base_path + dataset + exp_path + "/ddim/" + seed + '/ddpm/sample/'
    ddpm_reconstructions_path = base_path + dataset + exp_path + "/ddim/" + seed + '/ddpm/recons/'

    # load all images from the path and create a numpy array
    ddpm_samples = load_images_from_path(ddpm_samples_path)
    ddpm_reconstructions = load_images_from_path(ddpm_reconstructions_path)

    # precompute FID scores for generated and reconstructed images of DDPM

    # DDPM samples
    ddpm_stats_generations = save_fid_stats_as_dict(ddpm_samples, batch_size=32, device=device, dims=2048)
    train_FID_gen_ddpm = calculate_fid([data_stats_train, ddpm_stats_generations], batch_size=32, device=device, dims=2048)
    test_FID_gen_ddpm = calculate_fid([data_stats_test, ddpm_stats_generations], batch_size=32, device=device, dims=2048)
    cubicc_train_FID_generations_ddpm.append(train_FID_gen_ddpm)
    cubicc_test_FID_generations_ddpm.append(test_FID_gen_ddpm)

    # DDPM reconstructions
    ddpm_stats_reconstructions = save_fid_stats_as_dict(ddpm_reconstructions, batch_size=32, device=device, dims=2048)
    train_FID_recon_ddpm = calculate_fid([data_stats_train, ddpm_stats_reconstructions], batch_size=32, device=device, dims=2048)
    test_FID_recon_ddpm = calculate_fid([data_stats_test, ddpm_stats_reconstructions], batch_size=32, device=device, dims=2048)
    cubicc_train_FID_reconstructions_ddpm.append(train_FID_recon_ddpm)
    cubicc_test_FID_reconstructions_ddpm.append(test_FID_recon_ddpm)

In [ ]:
print("\nFID scores for DDPM samples compared to train set:\n", cubicc_train_FID_generations_ddpm)
print("\nFID scores for DDPM samples compared to test set:\n", cubicc_test_FID_generations_ddpm)
print("\nFID scores for DDPM reconstructions compared to train set:\n", cubicc_train_FID_reconstructions_ddpm)
print("\nFID scores for DDPM reconstructions compared to test set:\n", cubicc_test_FID_reconstructions_ddpm)

In [ ]:
# compute average and standard deviation of FID scores for DDPM 
print("CUBICC")
print("-----------------------------------")
print("Reconstructions")
print("\nAverage FID score for DDPM reconstructions compared to test set:", np.mean(cubicc_test_FID_reconstructions_ddpm))
print("\nStandard deviation FID score for DDPM reconstructions compared to test set:", np.std(cubicc_test_FID_reconstructions_ddpm))
print("-----------------------------------")
print("Generations")
print("\nAverage FID score for DDPM samples compared to test set:", np.mean(cubicc_test_FID_generations_ddpm))
print("\nStandard deviation FID score for DDPM samples compared to test set:", np.std(cubicc_test_FID_generations_ddpm))

# CelebA

In [ ]:
dataset = 'celeba/'

# use config of some trained cubicc model to load data correctly
ex_name = '20240303-144417_f2db4'

checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)

# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

# setup device, mps is for M1 or M2 macs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "mps"

# precompute or load fid scores for train and test
data_stats_train = get_precomputed_fid_scores_path(trainset.dataset.data, configs['data']['data_name'],
                                                subset="train", device=device)
data_stats_test = get_precomputed_fid_scores_path(testset.dataset.data, configs['data']['data_name'], subset="test",
                                                device=device)

# paths to the generated images
base_path = '../results_ICLR/'
exp_path = 'cond_on_path'
seed_list = ["seed_1", "seed_2", "seed_3", "seed_4", "seed_5", "seed_6", "seed_7", "seed_8", "seed_9", "seed_10"]

# lists to store FID scores for each seed
celeba_train_FID_generations_ddpm = []
celeba_test_FID_generations_ddpm = []
celeba_train_FID_reconstructions_ddpm = []
celeba_test_FID_reconstructions_ddpm = []

for i, seed in enumerate(seed_list):
    ddpm_samples_path = base_path + dataset + exp_path + "/ddim/" + seed + '/ddpm/sample/'
    ddpm_reconstructions_path = base_path + dataset + exp_path + "/ddim/" + seed + '/ddpm/recons/'

    # load all images from the path and create a numpy array
    ddpm_samples = load_images_from_path(ddpm_samples_path)
    ddpm_reconstructions = load_images_from_path(ddpm_reconstructions_path)

    # precompute FID scores for generated and reconstructed images of DDPM

    # DDPM samples
    ddpm_stats_generations = save_fid_stats_as_dict(ddpm_samples, batch_size=256, device=device, dims=2048)
    train_FID_gen_ddpm = calculate_fid([data_stats_train, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    test_FID_gen_ddpm = calculate_fid([data_stats_test, ddpm_stats_generations], batch_size=256, device=device, dims=2048)
    celeba_train_FID_generations_ddpm.append(train_FID_gen_ddpm)
    celeba_test_FID_generations_ddpm.append(test_FID_gen_ddpm)

    # DDPM reconstructions
    ddpm_stats_reconstructions = save_fid_stats_as_dict(ddpm_reconstructions, batch_size=256, device=device, dims=2048)
    train_FID_recon_ddpm = calculate_fid([data_stats_train, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    test_FID_recon_ddpm = calculate_fid([data_stats_test, ddpm_stats_reconstructions], batch_size=256, device=device, dims=2048)
    celeba_train_FID_reconstructions_ddpm.append(train_FID_recon_ddpm)
    celeba_test_FID_reconstructions_ddpm.append(test_FID_recon_ddpm)

In [ ]:
print("\nFID scores for DDPM samples compared to train set:\n", celeba_train_FID_generations_ddpm)
print("\nFID scores for DDPM samples compared to test set:\n", celeba_test_FID_generations_ddpm)
print("\nFID scores for DDPM reconstructions compared to train set:\n", celeba_train_FID_reconstructions_ddpm)
print("\nFID scores for DDPM reconstructions compared to test set:\n", celeba_test_FID_reconstructions_ddpm)

In [ ]:
# compute average and standard deviation of FID scores for DDPM 
print("CelebA")
print("-----------------------------------")
print("Reconstructions")
print("\nAverage FID score for DDPM reconstructions compared to test set:", np.mean(celeba_test_FID_reconstructions_ddpm))
print("\nStandard deviation FID score for DDPM reconstructions compared to test set:", np.std(celeba_test_FID_reconstructions_ddpm))
print("-----------------------------------")
print("Generations")
print("\nAverage FID score for DDPM samples compared to test set:", np.mean(celeba_test_FID_generations_ddpm))
print("\nStandard deviation FID score for DDPM samples compared to test set:", np.std(celeba_test_FID_generations_ddpm))